# Zehirli / Zehirsiz Böcek Veri Seti — Veri Toplama

Bu notebook, görüntü işleme dersi projesi için **zehirli ve zehirsiz böcek** fotoğraflarından oluşan bir veri setini otomatik olarak oluşturur.

**Kaynak:** [iNaturalist](https://www.inaturalist.org) — California Academy of Sciences ve National Geographic Society'nin ortak yürüttüğü vatandaş bilimi platformu. Kayıtlar GBIF'e (Küresel Biyoçeşitlilik Bilgi Sistemi) de aktarılır.

**Seçim kriterleri**
1. Yalnızca **Research Grade** gözlemler (fotoğraf, tarih ve konum bilgisi olan; tür tanımı topluluk tarafından doğrulanmış kayıtlar).
2. Yalnızca açık lisanslı fotoğraflar: **CC0, CC-BY, CC-BY-NC** (proje ticari değildir).
3. 24 takson grubu (12 zehirli, 12 zehirsiz), **aile / cins** seviyesinde seçildi; böylece her grupta çok sayıda farklı tür bulunur.
4. Yetişkini zararsız olan tırtıl gruplarında (*Thaumetopoea*, Megalopygidae, Limacodidae) yalnızca **larva** evresi alındı.
5. Gözlem başına 1 fotoğraf, grup başına en fazla **2.000** fotoğraf.

**Zehirli tanımımız:** İnsanı sokarak/ısırarak zehir veren veya dokunulduğunda zehirli kıl/salgı ile zarar veren böcekler.

Her fotoğrafın gözlem numarası, fotoğrafçısı, lisansı ve konumu `metadata.csv` dosyasına kaydedilir.

## 1. Ayarlar ve takson grupları

In [ ]:
import io, os, time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

API = "https://api.inaturalist.org/v1"
LISANSLAR = ["cc0", "cc-by", "cc-by-nc"]
MAKS_FOTO = 2000      # grup başına en fazla fotoğraf
MAKS_KENAR = 400      # fotoğraflar en uzun kenarı 400 piksel olacak şekilde küçültülür
KOK = "/kaggle/working/zehirli_bocek_veri"
LARVA = {"term_id": 1, "term_value_id": 6}   # iNaturalist "Life Stage = Larva" etiketi

GRUPLAR = [
    # takson,          sınıf,      sadece larva, açıklama
    ("Vespidae",       "zehirli",  False, "Eşek arıları ve hornetler (iğneli)"),
    ("Apis mellifera", "zehirli",  False, "Bal arısı (iğneli)"),
    ("Bombus",         "zehirli",  False, "Bombus arıları (iğneli)"),
    ("Solenopsis",     "zehirli",  False, "Ateş karıncaları (iğneli)"),
    ("Pompilidae",     "zehirli",  False, "Örümcek avcı arılar (iğneli)"),
    ("Mutillidae",     "zehirli",  False, "Kadife karıncalar (iğneli)"),
    ("Meloidae",       "zehirli",  False, "Kabarcık böcekleri (kantaridin salgısı)"),
    ("Paederus",       "zehirli",  False, "Paederus böcekleri (pederin, deride yanık)"),
    ("Reduviidae",     "zehirli",  False, "Suikastçı böcekler (zehirli ısırık)"),
    ("Thaumetopoea",   "zehirli",  True,  "Keseböceği tırtılları (zehirli kıllar)"),
    ("Megalopygidae",  "zehirli",  True,  "Kedi tırtılları (zehirli dikenler)"),
    ("Limacodidae",    "zehirli",  True,  "Sülük tırtılları (zehirli dikenler)"),
    ("Syrphidae",      "zehirsiz", False, "Sirfid sinekleri (arı taklitçisi)"),
    ("Muscidae",       "zehirsiz", False, "Ev sinekleri"),
    ("Tipulidae",      "zehirsiz", False, "Tipulid sinekleri (sivrisinek benzeri)"),
    ("Pieridae",       "zehirsiz", False, "Beyaz ve sarı kelebekler"),
    ("Lycaenidae",     "zehirsiz", False, "Mavi ve bakır kelebekler"),
    ("Odonata",        "zehirsiz", False, "Yusufçuklar ve kızböcekleri"),
    ("Acrididae",      "zehirsiz", False, "Çekirgeler"),
    ("Tettigoniidae",  "zehirsiz", False, "Çalı çekirgeleri"),
    ("Mantodea",       "zehirsiz", False, "Peygamber develeri"),
    ("Cicadidae",      "zehirsiz", False, "Ağustos böcekleri"),
    ("Cerambycidae",   "zehirsiz", False, "Teke böcekleri"),
    ("Chrysopidae",    "zehirsiz", False, "Altıngözler"),
]
gruplar = pd.DataFrame(GRUPLAR, columns=["takson", "sinif", "larva", "aciklama"])
gruplar["grup"] = gruplar.takson.str.replace(" ", "_")
gruplar

## 2. iNaturalist API yardımcıları ve gözlem sayıları

In [ ]:
oturum = requests.Session()
oturum.headers["User-Agent"] = "zehirli-bocek-ders-projesi (Kaggle notebook)"

def api(yol, **params):
    """iNaturalist API isteği; istek sınırına uymak için her istek arasında 1 sn bekler."""
    for deneme in range(5):
        time.sleep(1.0)
        try:
            r = oturum.get(f"{API}/{yol}", params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(5 * (deneme + 1))
    raise RuntimeError(f"API isteği başarısız: {yol} {params}")

def takson_bul(ad):
    sonuclar = api("taxa", q=ad, is_active="true", locale="tr", per_page=30)["results"]
    return next(t for t in sonuclar if t["name"] == ad)

def filtre(g):
    p = dict(taxon_id=g.taxon_id, quality_grade="research", photos="true",
             photo_license=",".join(LISANSLAR))
    if g.larva:
        p.update(LARVA)
    return p

taksonlar = [takson_bul(ad) for ad in gruplar.takson]
gruplar["taxon_id"] = [t["id"] for t in taksonlar]
gruplar["rank"] = [t["rank"] for t in taksonlar]
gruplar["ad_tr"] = [t.get("preferred_common_name", "") for t in taksonlar]
gruplar["gozlem"] = [api("observations", **filtre(g), per_page=0)["total_results"] for g in gruplar.itertuples()]
gruplar["alinacak"] = gruplar.gozlem.clip(upper=MAKS_FOTO)
print(gruplar.groupby("sinif").alinacak.sum(), "\nTOPLAM:", gruplar.alinacak.sum())
gruplar[["takson", "sinif", "rank", "ad_tr", "larva", "gozlem", "alinacak"]]

## 3. Fotoğraf listesi ve kaynak bilgileri (metadata)

In [ ]:
def gozlemleri_cek(g):
    satirlar, sayfa = [], 1
    while len(satirlar) < MAKS_FOTO:
        sonuc = api("observations", **filtre(g), per_page=200, page=sayfa,
                    order_by="id", order="desc")["results"]
        if not sonuc:
            break
        for o in sonuc:
            foto = next((p for p in o["photos"] if p.get("license_code") in LISANSLAR), None)
            if foto is None:
                continue
            satirlar.append(dict(
                sinif=g.sinif, grup=g.grup, obs_id=o["id"], photo_id=foto["id"],
                takson=o["taxon"]["name"], takson_rank=o["taxon"]["rank"],
                lisans=foto["license_code"], atif=foto.get("attribution"),
                kullanici=o["user"]["login"], tarih=o.get("observed_on"),
                yer=o.get("place_guess"),
                url=foto["url"].replace("/square.", "/medium."),
            ))
        sayfa += 1
    return satirlar[:MAKS_FOTO]

satirlar = []
for g in gruplar.itertuples():
    satirlar += gozlemleri_cek(g)
    print(f"{g.grup:15s} {g.sinif:9s} toplam liste: {len(satirlar)}")

meta = pd.DataFrame(satirlar).drop_duplicates("photo_id")
meta["dosya"] = ("images/" + meta.sinif + "/" + meta.grup + "/" + meta.photo_id.astype(str) + ".jpg")
print(meta.shape, "| farklı takson:", meta.takson.nunique())
meta.lisans.value_counts()

## 4. Fotoğrafları indirme ve küçültme

In [ ]:
def indir(url, hedef):
    if os.path.exists(hedef):
        return True
    for deneme in range(3):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                img = Image.open(io.BytesIO(r.content)).convert("RGB")
                img.thumbnail((MAKS_KENAR, MAKS_KENAR))
                img.save(hedef, "JPEG", quality=90)
                return True
        except Exception:
            pass
        time.sleep(2 * (deneme + 1))
    return False

for d in meta.dosya.map(lambda p: os.path.dirname(os.path.join(KOK, p))).unique():
    os.makedirs(d, exist_ok=True)

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    isler = {ex.submit(indir, u, os.path.join(KOK, p)): p for u, p in zip(meta.url, meta.dosya)}
    basarili = {isler[f]: f.result() for f in tqdm(as_completed(isler), total=len(isler), mininterval=30)}

meta["indirildi"] = meta.dosya.map(basarili)
meta = meta[meta.indirildi].drop(columns="indirildi")
boyut = sum(os.path.getsize(os.path.join(KOK, p)) for p in meta.dosya) / 1e9
print(f"İndirilen: {len(meta)} | Başarısız: {len(basarili) - len(meta)} | "
      f"Süre: {(time.time() - t0) / 60:.1f} dk | Boyut: {boyut:.2f} GB")

## 5. Kayıt dosyaları ve özet

In [ ]:
meta.to_csv(f"{KOK}/metadata.csv", index=False)
gruplar.to_csv(f"{KOK}/gruplar.csv", index=False)

lisans_sayilari = meta.lisans.value_counts().to_dict()
with open(f"{KOK}/KAYNAK.md", "w", encoding="utf-8") as f:
    f.write("# Kaynak ve Lisans\n\n")
    f.write("Fotoğraflar iNaturalist (https://www.inaturalist.org) Research Grade gözlemlerinden alınmıştır.\n")
    f.write(f"Toplam {len(meta)} fotoğraf, {meta.takson.nunique()} farklı takson.\n")
    f.write(f"Lisanslar: {lisans_sayilari}\n")
    f.write("Her fotoğrafın fotoğrafçısı ve lisansı metadata.csv dosyasının 'atif' ve 'lisans' sütunlarındadır.\n")

ozet = meta.groupby(["sinif", "grup"]).size().rename("foto").reset_index()
print(meta.sinif.value_counts())
renk = ozet.sinif.map({"zehirli": "tab:red", "zehirsiz": "tab:green"})
plt.figure(figsize=(14, 4))
plt.bar(ozet.grup, ozet.foto, color=renk)
plt.xticks(rotation=60, ha="right"); plt.ylabel("Fotoğraf sayısı")
plt.title("Gruplara göre fotoğraf sayısı (kırmızı: zehirli, yeşil: zehirsiz)")
plt.tight_layout(); plt.show()

In [ ]:
ornekler = meta.groupby("grup").head(2).reset_index(drop=True)
fig, axes = plt.subplots(6, 8, figsize=(20, 16))
for ax, s in zip(axes.flat, ornekler.itertuples()):
    ax.imshow(Image.open(os.path.join(KOK, s.dosya)))
    ax.set_title(f"{s.grup}\n({s.sinif})", fontsize=8, color="red" if s.sinif == "zehirli" else "green")
    ax.axis("off")
plt.tight_layout(); plt.show()